# Manual Experiment: Does attention entropy track question difficulty?

Samples questions from 3 sources with different, independent notions of difficulty coverage:
- **RACE++** (middle/high/race-c) — difficulty from which exam subset
- **OneStopQA** — difficulty from `level` (0/1/2 = elementary/intermediate/advanced), open-ended questions (not cloze)
- **SQuAD** — no difficulty tiers, used as a known-mostly-extractive baseline (should skew low entropy if the hypothesis holds)

For each sampled question: extract sentence-total attention distribution, plot it, show the entropy value, and manually judge whether it matches intuition. See `question_generation/docs/difficulty_steering_mechanisms.md` for the plan this feeds into.

## Contents

1. [Sample QA data](#load-sample)
2. [Define QA model](#qa-models)
3. [Extract attention and entropy](#extract-entropy)
4. [Inspect ranked examples (full text)](#inspect-passages)
5. [Inspect: literal vs inferential case](#minimal-pairs)
6. [Inspect: negation](#negation)
7. [Inspect: comparison and superlative](#comparison)
8. [Inspect: vocabulary/phrasing complexity](#vocab-complexity)
9. [Rate difficulty manually](#manual-ratings)

<a id="import"></a>
## 0. Imports and variables

In [33]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent.parent))

import random
import textwrap
import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset
from scipy.stats import spearmanr

from question_answering.qa_evaluator import QAEvaluator
from question_difficulty.methods.feature_based.difficulty_signals import AttentionDispersionSignal

random.seed(0)  # reproducible sampling across notebook re-runs

<a id="load-sample"></a>
## 1. Sample QA data

In [34]:
from collections import defaultdict

PASSAGES_PER_GROUP = 2  # passages per source group -- ALL (non-MC) questions on each are included
SAMPLES_CACHE = Path("entropy_review_samples.json")  # next to this notebook

# Same filter validate_difficulty_signals.py uses: RACE MC-verification
# questions ("Which of the following...") reference options we never load
# here, so the (passage, question) pair alone is meaningless for them --
# exclude at the source rather than let them show up as noise later.
_MC_PATTERNS = ("which of the following", "which one of the following", "which of these")

def _is_mc_question(question):
    q = question.lower()
    return any(p in q for p in _MC_PATTERNS)

def _pick_passages(by_passage, n_passages):
    """by_passage: {passage_text: [question, ...]}. Only passages with >=2
    questions are eligible (need multiple to compare within-passage)."""
    eligible = [p for p, qs in by_passage.items() if len(qs) >= 2]
    return random.sample(eligible, min(n_passages, len(eligible)))

def sample_race(subset, n_passages):
    ds = load_dataset("ehovy/race", subset, split="train")
    by_passage = defaultdict(list)
    for rec in ds:
        if _is_mc_question(rec["question"]):
            continue
        by_passage[rec["article"]].append(rec["question"])
    picked = _pick_passages(by_passage, n_passages)
    return [{"source": f"RACE-{subset}", "passage": p, "question": q}
            for p in picked for q in by_passage[p]]

def sample_race_c(n_passages):
    ds = load_dataset("tasksource/race-c", split="train")
    by_passage = defaultdict(list)
    for rec in ds:
        if _is_mc_question(rec["question"]):
            continue
        by_passage[rec["article"]].append(rec["question"])
    picked = _pick_passages(by_passage, n_passages)
    return [{"source": "RACE-C", "passage": p, "question": q}
            for p in picked for q in by_passage[p]]

def sample_onestopqa(level, n_passages):
    ds = load_dataset("malmaud/onestop_qa", split="train")
    level_name = {0: "elementary", 1: "intermediate", 2: "advanced"}[level]
    by_passage = defaultdict(list)
    for rec in ds:
        if rec["level"] != level or _is_mc_question(rec["question"]):
            continue
        by_passage[rec["paragraph"]].append(rec["question"])
    picked = _pick_passages(by_passage, n_passages)
    return [{"source": f"OneStopQA-{level_name}", "passage": p, "question": q}
            for p in picked for q in by_passage[p]]

def sample_squad(n_passages):
    ds = load_dataset("rajpurkar/squad", split="train")
    by_passage = defaultdict(list)
    for rec in ds:
        if _is_mc_question(rec["question"]):
            continue
        by_passage[rec["context"]].append(rec["question"])
    picked = _pick_passages(by_passage, n_passages)
    return [{"source": "SQuAD", "passage": p, "question": q}
            for p in picked for q in by_passage[p]]

import json

if SAMPLES_CACHE.exists():
    samples = json.loads(SAMPLES_CACHE.read_text())
    print(f"Loaded {len(samples)} CACHED samples from {SAMPLES_CACHE} (delete this file to re-sample)")
else:
    samples = []
    samples += sample_race("middle", PASSAGES_PER_GROUP)
    samples += sample_race("high", PASSAGES_PER_GROUP)
    samples += sample_race_c(PASSAGES_PER_GROUP)
    samples += sample_onestopqa(0, PASSAGES_PER_GROUP)
    samples += sample_onestopqa(1, PASSAGES_PER_GROUP)
    samples += sample_onestopqa(2, PASSAGES_PER_GROUP)
    samples += sample_squad(PASSAGES_PER_GROUP)
    SAMPLES_CACHE.write_text(json.dumps(samples, indent=2, ensure_ascii=False))
    print(f"Sampled {len(samples)} fresh examples, cached to {SAMPLES_CACHE} for future sessions")

n_unique_passages = len({(s["source"], s["passage"]) for s in samples})
print(f"{n_unique_passages} unique passages, {len(samples)} total questions "
      f"({len(samples) / n_unique_passages:.1f} questions/passage avg)")
for s in samples[:3]:
    print(f"  [{s['source']}] {s['question'][:70]}")

Loaded 52 CACHED samples from entropy_review_samples.json (delete this file to re-sample)
14 unique passages, 52 total questions (3.7 questions/passage avg)
  [RACE-middle] This is an orange   _  .
  [RACE-middle] The pen is   _  .
  [RACE-middle] _   pencils are in the pencil case.


<a id="qa-models"></a>
## 2. Define QA model

All verified to actually load (never trust a model ID without checking --
`mrm8488/distilroberta-base-finetuned-squad`, previously used in this
project, currently 404s on HF Hub; excluded here, needs a separate fix in
`question_answering/docs/qa_model_battery.md`).

`vasudevgupta/bigbird-roberta-natural-questions` and
`google/bigbird-base-trivia-itc` (Natural Questions / TriviaQA -- genuinely
non-SQuAD) were also considered but excluded: neither ships `safetensors`
weights, and the installed `transformers` now refuses to load legacy
`pytorch_model.bin` checkpoints without `torch>=2.6` (CVE-2025-32434). Not
worth a project-wide torch upgrade just for this comparison.

`consciousAI/question-answering-roberta-base-s-v2` is the one non-SQuAD-named
model that actually loads and runs cleanly (standard RoBERTa, same dense
attention as the others) -- included below, though its exact training data
mix isn't independently confirmable from here (no model-card browsing access
in this environment), only that it isn't named/marketed as SQuAD-only like
the others.

Loads all 4 up front so any cell below can pick which one to use for
comparison, without re-downloading mid-session.

In [35]:
from question_difficulty.methods.feature_based.difficulty_signals import QuestionAnswerSimilaritySignal

QA_MODEL_CANDIDATES = {
    "roberta-base-squad2": "deepset/roberta-base-squad2",       # standard baseline
    "deberta-v3-base-squad2": "deepset/deberta-v3-base-squad2", # stronger
    "distilbert-squad": "distilbert-base-cased-distilled-squad", # weaker/smaller, for contrast
    "roberta-nonsquad": "consciousAI/question-answering-roberta-base-s-v2", # not SQuAD-named
}

signals = {name: AttentionDispersionSignal(qa_model_name=model_id)
           for name, model_id in QA_MODEL_CANDIDATES.items()}

QA_MODEL = "deberta-v3-base-squad2"

signal = signals[QA_MODEL]
qa_evaluator = QAEvaluator()
qa_sim_signal = QuestionAnswerSimilaritySignal()

LAYER = 11

def predict_answer(passage, question):
    """Runs the actual QA model's extractive prediction (not just attention) --
    lets us check, per row, whether high entropy means "genuinely reasoning
    more broadly" or "the model is lost and guessing badly". Only depends on
    `signal` (this cell) -- reused by every section below, independently of
    each other."""
    tokenizer, model, torch = signal.tokenizer, signal.model, signal.torch
    enc = tokenizer(question, passage, max_length=signal.max_length,
                     truncation="only_second", return_offsets_mapping=True, return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    sequence_ids = enc.sequence_ids(0)
    passage_positions = [i for i, s in enumerate(sequence_ids) if s == 1]

    with torch.no_grad():
        out = model(**enc)

    start_probs = torch.softmax(out.start_logits[0], dim=-1)
    end_probs = torch.softmax(out.end_logits[0], dim=-1)
    start_idx = max(passage_positions, key=lambda i: start_probs[i].item())
    end_idx = max(passage_positions, key=lambda i: end_probs[i].item())
    if end_idx < start_idx:
        end_idx = start_idx
    char_start, char_end = offsets[start_idx][0], offsets[end_idx][1]
    confidence = start_probs[start_idx].item() * end_probs[end_idx].item()
    return passage[char_start:char_end].strip(), confidence

def _score_question(passage, question, answer):
    detail = signal.get_sentence_distribution(passage, question, layer=LAYER)
    overlap = qa_sim_signal.compute(passage, question, answer)["question_answer_overlap_coef"]
    pred, conf = predict_answer(passage, question)
    f1 = qa_evaluator.token_f1(pred, answer)
    return {
        "question": question, "answer": answer,
        "tok_entropy_norm": detail["tok_entropy_norm"], "sent_entropy_norm": detail["entropy_norm"],
        "overlap": overlap, "pred": pred, "conf": conf, "f1": f1,
        "correct": qa_evaluator.is_correct(pred, answer),
    }

def compare_question_pair(passage, label_a, question_a, answer_a, label_b, question_b, answer_b):
    """Same print/score pattern used by every "Inspect: A vs. B" section
    below (literal vs. inferential, negation, vocabulary) -- factored out
    here so each section reuses one implementation instead of duplicating
    the loop body. Only depends on names defined in this cell."""
    compare_question_set(passage, [(label_a, question_a, answer_a), (label_b, question_b, answer_b)])

def compare_question_set(passage, items):
    """Like compare_question_pair but for 3+ conditions on the SAME passage
    printed together in one block, with deltas against the first item
    (treated as the baseline) -- e.g. literal + comparison + superlative
    side by side in §7, instead of two separate two-way printouts. items:
    list of (label, question, answer) tuples."""
    computed = [{"label": label, **_score_question(passage, question, answer)}
                for label, question, answer in items]

    print("=" * 100)
    print(f"PASSAGE: {passage}")
    print()
    for c in computed:
        print(f"  {c['label']:12s} Q: {c['question']}")
        print(f"               gold={c['answer']!r}")
        print(f"               tok_entropy_norm={c['tok_entropy_norm']:.3f}  sent_entropy_norm={c['sent_entropy_norm']:.3f}  question_answer_overlap_coef={c['overlap']:.3f}")
        print(f"               predicted_answer={c['pred']!r}  confidence={c['conf']:.3f}  f1={c['f1']:.3f}  {'CORRECT' if c['correct'] else 'WRONG'}")
        print()

    base = computed[0]
    print(f"  deltas vs {base['label']}:")
    for c in computed[1:]:
        print(f"    {c['label']:12s} tok_entropy_norm={c['tok_entropy_norm']-base['tok_entropy_norm']:+.3f}  "
              f"sent_entropy_norm={c['sent_entropy_norm']-base['sent_entropy_norm']:+.3f}  "
              f"overlap={c['overlap']-base['overlap']:+.3f}  confidence={c['conf']-base['conf']:+.3f}  f1={c['f1']-base['f1']:+.3f}")
    print()

<a id="extract-entropy"></a>
## 3. Extract attention and entropy

Single layer, from `QA_MODEL`.

In [ ]:
FIXED_QUESTION = "What is the topic of the passage?"

results = []
for s in samples:
    detail = signal.get_sentence_distribution(s["passage"], s["question"], layer=LAYER)
    detail["num_sentences"] = len(detail["sentences"])
    detail["predicted_answer"], detail["confidence"] = predict_answer(s["passage"], s["question"])
    results.append({**s, **detail})

# Sanity check: raw entropy is bounded by log(N) (N = num outcomes: sentences
# for sent_entropy, tokens for tok_entropy), so more outcomes = higher
# entropy CEILING regardless of how spread attention actually is. If raw
# entropy correlates with length, that's the length confound, not a
# difficulty signal. entropy_norm (= entropy / log(N)) is meant to correct
# for it -- check whether it does, for both levels. Computed on the REAL
# questions only, before the fixed-probe rows below get appended, so the
# probe (same question repeated per passage) doesn't skew this.
lengths_sent = [r["num_sentences"] for r in results]
lengths_tok = [r["num_tokens"] for r in results]
rho_sent_raw, _ = spearmanr(lengths_sent, [r["entropy"] for r in results])
rho_sent_norm, _ = spearmanr(lengths_sent, [r["entropy_norm"] for r in results])
rho_tok_raw, _ = spearmanr(lengths_tok, [r["tok_entropy"] for r in results])
rho_tok_norm, _ = spearmanr(lengths_tok, [r["tok_entropy_norm"] for r in results])
print(f"Spearman(n_sent, sent raw entropy)  = {rho_sent_raw:.3f}   Spearman(n_sent, sent norm entropy)  = {rho_sent_norm:.3f}")
print(f"Spearman(n_tok,  tok  raw entropy)  = {rho_tok_raw:.3f}   Spearman(n_tok,  tok  norm entropy)  = {rho_tok_norm:.3f}")
print("(norm columns should be closer to 0 than raw if normalization worked)")
print()

# Fixed-question probe: same question ("What is the topic of the passage?")
# against every unique passage. Source tagged "Probe-question-<original
# source>" (e.g. "Probe-question-RACE-middle") so the passage's origin is
# visible at a glance -- appended into `results` so it's sortable/rateable/
# viewable through the same IDX machinery as everything else below.
seen_passages = {}
for s in samples:
    key = (s["source"], s["passage"])
    seen_passages.setdefault(key, s["passage"])
for (source, passage) in seen_passages:
    detail = signal.get_sentence_distribution(passage, FIXED_QUESTION, layer=LAYER)
    detail["num_sentences"] = len(detail["sentences"])
    detail["predicted_answer"], detail["confidence"] = predict_answer(passage, FIXED_QUESTION)
    results.append({"source": f"Probe-question-{source}", "passage": passage, "question": FIXED_QUESTION, **detail})

results.sort(key=lambda r: r["tok_entropy_norm"] if r.get("tok_entropy_norm") is not None else -1)
print(f"Sorted low -> high TOKEN normalized entropy ({QA_MODEL}, layer {LAYER}):")
for idx, r in enumerate(results):
    print(f"[{idx:>3}] tok_entropy_norm={r['tok_entropy_norm']:.3f}  sent_entropy_norm={r['entropy_norm']:.3f}  n_tok={r['num_tokens']:>4}  n_sent={r['num_sentences']:>3}  confidence={r['confidence']:.3f}")
    print(f"      [{r['source']}]  {r['question']}")
    print(f"      predicted_answer={r['predicted_answer']!r}")

<a id="inspect-passages"></a>
## 4. Inspect ranked examples (full text)

In [ ]:
PAGE_IDXS = [7, 18, 41, 51]

for idx in PAGE_IDXS:
    r = results[idx]
    print("=" * 100)
    print(f"[{idx}] tok_entropy_norm={r['tok_entropy_norm']:.3f}  sent_entropy_norm={r['entropy_norm']:.3f}  n_tok={r['num_tokens']}  n_sent={r['num_sentences']}  confidence={r['confidence']:.3f}  [{r['source']}]")
    print(f"Q: {r['question']}")
    print(f"predicted_answer: {r['predicted_answer']!r}")
    print()
    print(f"PASSAGE:\n{r['passage']}")
    print()

<a id="minimal-pairs"></a>
## 5. Inspect: literal vs. inferential case

The signals above measure ANSWER LOCALIZATION difficulty -- how much of the
passage the model has to spread across to find where the evidence is. They
say nothing about INFERENTIAL/REASONING difficulty -- how much cognitive
work is needed to go from that evidence to the answer once it's found.

Two questions that point at the exact same evidence sentence but require
different reasoning (a literal "what" restating the fact vs. an inferential
"why" requiring the reader to reason about it) should attend to roughly the
SAME region of the passage -- an extractive QA model has no mechanism to
"look elsewhere" just because the question demands more reasoning. If their
entropy comes out near-identical despite very different subjective
difficulty, that CONFIRMS the blind spot: entropy alone can't distinguish
literal from inferential comprehension (Barrett's Taxonomy terms) -- it's
one axis of difficulty (locatability), not the whole thing.

In [39]:
# signal, predict_answer(), qa_evaluator, qa_sim_signal, and LAYER are all
# defined in §2 above -- this section only depends on §1+§2, not §3's
# sample extraction.

# Each pair: same passage, one question is a literal restatement and the
# other requires reasoning about that evidence. literal_answer and
# inferential_answer are scored SEPARATELY (not one shared "answer") --
# the natural complete answer to "why" is often longer/different than the
# natural answer to "what" once the passage has any real causal chain to
# explain (a single shared span only worked for the 2 shortest passages
# below, where the literal fact IS the complete causal explanation).
MINIMAL_PAIRS = [
    {
        # SHORT (~30 words)
        "passage": "Lily always carries a spare phone charger in her bag. Her phone battery tends to die quickly during long days at work.",
        "literal_question": "What does Lily always carry in her bag?",
        "literal_answer": "a spare phone charger",
        "inferential_question": "Why won't Lily be stuck with a dead phone during a long day at work?",
        "inferential_answer": "a spare phone charger",
    },
    {
        # SHORT (~50 words)
        "passage": "Every school day, Maria wakes up early. She needs to go to school at 9AM, so she prepares her books and uniform the night before. Her mother drops her off on the way to work.",
        "literal_question": "What does she need to do at 9AM?",
        "literal_answer": "go to school",
        "inferential_question": "Why can't she wake up at 10AM?",
        "inferential_answer": "she needs to go to school at 9AM",
    },
    {
        # MEDIUM (~95 words)
        "passage": "Every winter, the small town of Millbrook loses power for a few hours during storms. Last year, the local government installed a backup generator at the community center to keep the building running during outages. The generator can supply electricity for up to 12 hours before needing more fuel. Residents can charge their phones and stay warm there when their home power goes out. Some elderly residents rely on the center to keep medical equipment running. The town is now considering installing solar panels to reduce fuel costs.",
        "literal_question": "What did the local government install at the community center?",
        "literal_answer": "a backup generator",
        "inferential_question": "Why can Millbrook residents stay warm and charge their phones at the community center during a power outage?",
        "inferential_answer": "the generator can supply electricity",
    },
    {
        # LONG (~190 words)
        "passage": "In 2015, a small biotech company in Boston began testing a new type of bandage designed for chronic wounds. Unlike traditional bandages, this one contained a thin layer of sensors that measured moisture, temperature, and bacterial growth in real time. The data was sent wirelessly to a smartphone app, allowing doctors to monitor a patient's healing progress without requiring a clinic visit. Early trials showed that patients using the smart bandage healed nearly 20 percent faster than those using standard dressings. Doctors were able to adjust treatment plans within a day of an infection appearing, rather than waiting for the patient's next scheduled appointment. The bandage costs significantly more than a regular one, which has slowed its adoption in hospitals with tight budgets. Even so, several insurance companies have begun covering the cost for patients with diabetes, who are especially prone to slow-healing wounds. The company hopes to expand production and lower the price within the next two years.",
        "literal_question": "What three things did the sensors in the bandage measure?",
        "literal_answer": "moisture, temperature, and bacterial growth",
        "inferential_question": "Why were doctors able to adjust treatment plans within a day of an infection appearing?",
        "inferential_answer": "sensors measured moisture, temperature, and bacterial growth in real time",
    },
    {
        # VERY LONG (~230 words)
        "passage": "When Elena took over as principal of Riverside Elementary three years ago, nearly a third of the school's students were missing more than two weeks of class every year. After interviewing families, she discovered that many parents worked early shifts and had no way to get their children to school on time, since the nearest bus stop was almost two miles away. Elena partnered with a local transit company to add a new bus route that stopped directly in front of the school at 7:15 each morning. She also arranged for breakfast to be served starting at 7:00, so students arriving early would not have to wait outside in the cold. Within the first semester, chronic absenteeism dropped from 32 percent to 19 percent. Teachers reported that students who used to miss the first class of the day were now consistently present, which allowed lessons to build on each other more smoothly. The district was impressed enough that it approved funding to extend the new bus route to two neighboring schools the following year. Some parents initially worried that the earlier start time would be difficult for younger children, but a survey conducted at the end of the year found that most families adjusted within the first month. Elena says the biggest lesson from the program was that a single logistical barrier, transportation, was quietly undermining years of curriculum improvements the school had already made.",
        "literal_question": "What time does the new bus route stop in front of the school?",
        "literal_answer": "7:15",
        "inferential_question": "Why did chronic absenteeism drop after Elena introduced the new bus route?",
        "inferential_answer": "the new bus route gave students a way to get to school on time",
    },
]

for pair in MINIMAL_PAIRS:
    compare_question_pair(
        pair["passage"],
        "LITERAL", pair["literal_question"], pair["literal_answer"],
        "INFERENTIAL", pair["inferential_question"], pair["inferential_answer"],
    )

PASSAGE: Lily always carries a spare phone charger in her bag. Her phone battery tends to die quickly during long days at work.

  LITERAL      Q: What does Lily always carry in her bag?
               gold='a spare phone charger'
               tok_entropy_norm=0.848  sent_entropy_norm=0.585  question_answer_overlap_coef=0.000
               predicted_answer='a spare phone charger'  confidence=0.714  f1=1.000  CORRECT

  INFERENTIAL  Q: Why won't Lily be stuck with a dead phone during a long day at work?
               gold='a spare phone charger'
               tok_entropy_norm=0.949  sent_entropy_norm=0.999  question_answer_overlap_coef=0.333
               predicted_answer='spare phone charger'  confidence=0.000  f1=0.857  CORRECT

  deltas vs LITERAL:
    INFERENTIAL  tok_entropy_norm=+0.100  sent_entropy_norm=+0.414  overlap=+0.333  confidence=-0.714  f1=-0.143

PASSAGE: Every school day, Maria wakes up early. She needs to go to school at 9AM, so she prepares her books and unifor

<a id="negation"></a>
## 6. Inspect: negation

A different cognitive operation from causal "why" bridging: verifying
ABSENCE across the passage rather than locating a stated fact. Deliberately
NOT phrased as MC-style "which of the following is NOT true" (those get
filtered out of the main sample in §1 -- their real content lives in
options we never load, so the bare question is meaningless on its own).
Instead each passage below states the negative fact explicitly, so there's
a genuine extractable answer for the negation question too, not just an
absence with nothing to point at.

In [38]:
# compare_question_pair(), signal, predict_answer(), qa_evaluator, and
# qa_sim_signal are all defined in §2 -- independent of §3/§5.

NEGATION_PAIRS = [
    {
        "passage": "The new employee handbook covers vacation policy, sick leave, and health insurance. It does not include any information about retirement benefits, since those are managed by a separate HR portal.",
        "positive_question": "What three topics does the employee handbook cover?",
        "positive_answer": "vacation policy, sick leave, and health insurance",
        "negation_question": "What topic does the employee handbook NOT include?",
        "negation_answer": "retirement benefits",
    },
    {
        # VERY SHORT (~30 words)
        "passage": "The gym's monthly membership includes access to the weight room and cardio machines, but it does not include personal training sessions.",
        "positive_question": "What does the gym membership include?",
        "positive_answer": "the weight room and cardio machines",
        "negation_question": "What does the gym membership NOT include?",
        "negation_answer": "personal training sessions",
    },
    {
        # SHORT (~35 words)
        "passage": "The cafe's lunch menu offers soup, salad, and a daily sandwich special. It does not offer any hot entrees until dinner service begins at 5PM.",
        "positive_question": "What three items does the cafe's lunch menu offer?",
        "positive_answer": "soup, salad, and a daily sandwich special",
        "negation_question": "What does the cafe's lunch menu NOT offer?",
        "negation_answer": "hot entrees",
    },
    {
        # MEDIUM (~90 words)
        "passage": "The basic travel insurance plan covers trip cancellation, lost luggage, and emergency medical expenses abroad. It does not cover pre-existing medical conditions unless the traveler purchases an additional waiver within 14 days of booking. Many customers are surprised to learn this exclusion applies even to well-controlled conditions like diabetes or asthma. Travelers with chronic illnesses are strongly advised to read the fine print before relying on the basic plan alone.",
        "positive_question": "What three things does the basic travel insurance plan cover?",
        "positive_answer": "trip cancellation, lost luggage, and emergency medical expenses abroad",
        "negation_question": "What does the basic travel insurance plan NOT cover unless a waiver is purchased?",
        "negation_answer": "pre-existing medical conditions",
    },
    {
        # LONG (~185 words)
        "passage": "Starting next month, the city's curbside recycling program will accept paper, cardboard, glass bottles, and most plastics labeled 1 through 5. City officials spent the past year studying which materials could realistically be processed at the local sorting facility without expensive upgrades. The program will not accept plastic bags, styrofoam, or any electronics, since the facility lacks the equipment to safely process them and past attempts led to jammed machinery. Residents who want to dispose of electronics can instead drop them off at the county hazardous waste center, which accepts them free of charge on the first Saturday of every month. City officials estimate that clearly excluding these problematic materials from curbside pickup will reduce contamination rates at the sorting facility by nearly 40 percent, based on data from similar programs in neighboring cities.",
        "positive_question": "What four types of materials will the curbside recycling program accept?",
        "positive_answer": "paper, cardboard, glass bottles, and most plastics labeled 1 through 5",
        "negation_question": "What three things will the curbside recycling program NOT accept?",
        "negation_answer": "plastic bags, styrofoam, or any electronics",
    },
]

for pair in NEGATION_PAIRS:
    compare_question_pair(
        pair["passage"],
        "POSITIVE", pair["positive_question"], pair["positive_answer"],
        "NEGATION", pair["negation_question"], pair["negation_answer"],
    )

PASSAGE: The new employee handbook covers vacation policy, sick leave, and health insurance. It does not include any information about retirement benefits, since those are managed by a separate HR portal.

  POSITIVE     Q: What three topics does the employee handbook cover?
               gold='vacation policy, sick leave, and health insurance'
               tok_entropy_norm=0.832  sent_entropy_norm=0.683  question_answer_overlap_coef=0.000
               predicted_answer='vacation policy, sick leave, and health insurance'  confidence=0.999  f1=1.000  CORRECT

  NEGATION     Q: What topic does the employee handbook NOT include?
               gold='retirement benefits'
               tok_entropy_norm=0.847  sent_entropy_norm=0.823  question_answer_overlap_coef=0.000
               predicted_answer='retirement benefits'  confidence=0.999  f1=1.000  CORRECT

  deltas vs POSITIVE:
    NEGATION     tok_entropy_norm=+0.015  sent_entropy_norm=+0.140  overlap=+0.000  confidence=+0.000  f1=+

<a id="comparison"></a>
## 7. Inspect: comparison and superlative

Two distinct 2-hop-or-more operations, both against the same literal
baseline per pair:
- **comparison**: retrieve two named facts and compare them (greater/less,
  before/after) -- no causality, no absence-checking.
- **superlative**: evaluate ALL candidates (3 per passage here) to find the
  extremum ("most", "longest", "first"). Genuinely harder than binary
  comparison -- can't just check two named options, has to actually rank
  everything stated.

This isolates "hop count" from "causal reasoning depth", which the
literal-vs-inferential pairs in §5 conflated (their inferential questions
always required BOTH more hops AND causal reasoning together, so we
couldn't tell which one entropy was actually responding to).

In [37]:
# compare_question_set() etc. are defined in §2 -- independent of §3/§5/§6.

# Each pair now has THREE entities, so comparison (binary, pick between two
# named ones) and superlative (must evaluate all three to find the extremum)
# are genuinely different operations, not the same thing at different scale.
COMPARISON_PAIRS = [
    {
        "passage": "The Riverside factory produced 1,200 units in January and 1,850 units in March. The Lakeside factory produced 1,500 units in January and 1,600 units in March. The Hillcrest factory produced 1,300 units in January and 1,420 units in March.",
        "literal_question": "How many units did the Riverside factory produce in March?",
        "literal_answer": "1,850",
        "comparison_question": "Which factory produced more units in March, Riverside or Lakeside?",
        "comparison_answer": "Riverside",
        "superlative_question": "Which factory produced the most units in March?",
        "superlative_answer": "Riverside",
    },
    {
        # SHORT (~50 words), numeric magnitude
        "passage": "Sarah and her friend had a marathon yesteday afterwork. Sarah finished the marathon in 3 hours and 45 minutes. Her teammate Priya finished in 3 hours and 52 minutes. Their teammate Wei finished in 3 hours and 38 minutes. The marathon was all first time particpating for all of them.",
        "literal_question": "How long did Sarah take to finish the marathon?",
        "literal_answer": "3 hours and 45 minutes",
        "comparison_question": "Who finished the marathon faster, Sarah or Priya?",
        "comparison_answer": "Sarah",
        "superlative_question": "Who finished the marathon fastest among the three teammates?",
        "superlative_answer": "Wei",
    },
    {
        # MEDIUM (~80 words), temporal precedence instead of magnitude
        "passage": "Museum Charsm is one of the oldest museums in the country. It had been in private control for many years until 1994. The museum's east wing opened in 1998. The west wing opened in 2006, after a major renovation project. The south wing, the newest addition, opened in 2015. Nowadays, visitors can freely enter and wander all the museum with a very affordable cost of 15e. The museum is open from 9 to 6 everyday except Monday. ",
        "literal_question": "When did the museum's west wing open?",
        "literal_answer": "2006",
        "comparison_question": "Which wing of the museum opened first, the east wing or the west wing?",
        "comparison_answer": "the east wing",
        "superlative_question": "Which wing of the museum opened most recently?",
        "superlative_answer": "the south wing",
    },
    {
        # MEDIUM (~110 words)
        "passage": "The Zenith X200 smartphone has a battery life of 14 hours and costs $699. Its main competitor, the Aurora S5, has a battery life of 18 hours but costs $799. A newer budget option, the Nova Lite, has a battery life of 20 hours and costs only $549. Reviewers noted that despite its higher price, the Aurora S5 has become popular among frequent travelers who value longer battery life over cost savings. The Zenith X200 remains popular with budget-conscious buyers who charge their phones more frequently throughout the day. The Nova Lite has quickly gained a following among budget shoppers who also want long battery life.",
        "literal_question": "What is the battery life of the Aurora S5?",
        "literal_answer": "18 hours",
        "comparison_question": "Which phone has a longer battery life, the Zenith X200 or the Aurora S5?",
        "comparison_answer": "the Aurora S5",
        "superlative_question": "Which of the three phones has the longest battery life?",
        "superlative_answer": "the Nova Lite",
    },
    {
        # LONG (~180 words), comparison embedded in a causal-explanation-heavy passage
        "passage": "A recent health department report compared emergency room wait times at three hospitals in the same county. At Northside General, the average wait time for non-critical patients was 94 minutes, and the hospital employed 12 full-time emergency physicians. At Southbrook Medical Center, the average wait time was 61 minutes, despite employing only 9 full-time emergency physicians. At Eastview Regional, the average wait time was 108 minutes, the longest of the three, even though the hospital employed 14 full-time emergency physicians. Investigators found that Southbrook's shorter wait times were largely due to a triage software system introduced two years earlier, which allowed nurses to route minor cases to a fast-track unit staffed by physician assistants. Northside had piloted a similar system the previous year but discontinued it after complaints about inconsistent case routing. Eastview has not yet implemented any triage software, and administrators say budget constraints have delayed the project. The health department recommended that Northside and Eastview both revisit the triage software with updated training protocols before their next budget cycles.",
        "literal_question": "How many full-time emergency physicians did Northside General employ?",
        "literal_answer": "12",
        "comparison_question": "Which hospital had shorter emergency room wait times, Northside General or Southbrook Medical Center?",
        "comparison_answer": "Southbrook Medical Center",
        "superlative_question": "Which of the three hospitals had the longest emergency room wait time?",
        "superlative_answer": "Eastview Regional",
    },
]

for pair in COMPARISON_PAIRS:
    compare_question_set(pair["passage"], [
        ("LITERAL", pair["literal_question"], pair["literal_answer"]),
        ("COMPARISON", pair["comparison_question"], pair["comparison_answer"]),
        ("SUPERLATIVE", pair["superlative_question"], pair["superlative_answer"]),
    ])

PASSAGE: The Riverside factory produced 1,200 units in January and 1,850 units in March. The Lakeside factory produced 1,500 units in January and 1,600 units in March. The Hillcrest factory produced 1,300 units in January and 1,420 units in March.

  LITERAL      Q: How many units did the Riverside factory produce in March?
               gold='1,850'
               tok_entropy_norm=0.749  sent_entropy_norm=0.468  question_answer_overlap_coef=0.000
               predicted_answer='1,850'  confidence=0.996  f1=1.000  CORRECT

  COMPARISON   Q: Which factory produced more units in March, Riverside or Lakeside?
               gold='Riverside'
               tok_entropy_norm=0.818  sent_entropy_norm=0.943  question_answer_overlap_coef=1.000
               predicted_answer='Hillcrest'  confidence=0.479  f1=0.000  WRONG

  SUPERLATIVE  Q: Which factory produced the most units in March?
               gold='Riverside'
               tok_entropy_norm=0.865  sent_entropy_norm=0.922  question_an

<a id="vocab-complexity"></a>
## 8. Inspect: vocabulary/phrasing complexity (cognitive operation held constant)

Same passage, same evidence, same reasoning operation (plain literal
recall) -- ONLY the wording of the question changes, plain vs. dense/
formal. If entropy shifts here despite zero change in reasoning demand,
that's direct evidence entropy is partly reacting to surface/linguistic
complexity, not pure cognitive difficulty -- the exact confound raised
earlier: "difficulty could include grammar or vocab of the question...
but cognitive difficulty only" is a separate claim, and this is the test
that isolates it.

In [28]:
VOCAB_PAIRS = [
    {
        # SHORT (~30 words)
        "passage": "Lily always carries a spare phone charger in her bag. Her phone battery tends to die quickly during long days at work.",
        "plain_question": "What does Lily always carry in her bag?",
        "plain_answer": "a spare phone charger",
        "dense_question": "What accessory does Lily routinely transport within her bag?",
        "dense_answer": "a spare phone charger",
    },
    {
        # SHORT (~50 words)
        "passage": "Every school day, Maria wakes up early. She needs to go to school at 9AM, so she prepares her books and uniform the night before. Her mother drops her off on the way to work.",
        "plain_question": "What does she need to do at 9AM?",
        "plain_answer": "go to school",
        "dense_question": "What obligation necessitates her early departure at 9AM?",
        "dense_answer": "go to school",
    },
    {
        # MEDIUM (~95 words)
        "passage": "Every winter, the small town of Millbrook loses power for a few hours during storms. Last year, the local government installed a backup generator at the community center to keep the building running during outages. The generator can supply electricity for up to 12 hours before needing more fuel. Residents can charge their phones and stay warm there when their home power goes out. Some elderly residents rely on the center to keep medical equipment running. The town is now considering installing solar panels to reduce fuel costs.",
        "plain_question": "What did the local government install at the community center?",
        "plain_answer": "a backup generator",
        "dense_question": "What apparatus did municipal authorities procure and install at the communal facility?",
        "dense_answer": "a backup generator",
    },
    {
        # LONG (~190 words)
        "passage": "In 2015, a small biotech company in Boston began testing a new type of bandage designed for chronic wounds. Unlike traditional bandages, this one contained a thin layer of sensors that measured moisture, temperature, and bacterial growth in real time. The data was sent wirelessly to a smartphone app, allowing doctors to monitor a patient's healing progress without requiring a clinic visit. Early trials showed that patients using the smart bandage healed nearly 20 percent faster than those using standard dressings. Doctors were able to adjust treatment plans within a day of an infection appearing, rather than waiting for the patient's next scheduled appointment. The bandage costs significantly more than a regular one, which has slowed its adoption in hospitals with tight budgets. Even so, several insurance companies have begun covering the cost for patients with diabetes, who are especially prone to slow-healing wounds. The company hopes to expand production and lower the price within the next two years.",
        "plain_question": "What three things did the sensors in the bandage measure?",
        "plain_answer": "moisture, temperature, and bacterial growth",
        "dense_question": "What three physiological parameters were quantified by the embedded sensor array?",
        "dense_answer": "moisture, temperature, and bacterial growth",
    },
    {
        # VERY LONG (~230 words)
        "passage": "When Elena took over as principal of Riverside Elementary three years ago, nearly a third of the school's students were missing more than two weeks of class every year. After interviewing families, she discovered that many parents worked early shifts and had no way to get their children to school on time, since the nearest bus stop was almost two miles away. Elena partnered with a local transit company to add a new bus route that stopped directly in front of the school at 7:15 each morning. She also arranged for breakfast to be served starting at 7:00, so students arriving early would not have to wait outside in the cold. Within the first semester, chronic absenteeism dropped from 32 percent to 19 percent. Teachers reported that students who used to miss the first class of the day were now consistently present, which allowed lessons to build on each other more smoothly. The district was impressed enough that it approved funding to extend the new bus route to two neighboring schools the following year. Some parents initially worried that the earlier start time would be difficult for younger children, but a survey conducted at the end of the year found that most families adjusted within the first month. Elena says the biggest lesson from the program was that a single logistical barrier, transportation, was quietly undermining years of curriculum improvements the school had already made.",
        "plain_question": "What time does the new bus route stop in front of the school?",
        "plain_answer": "7:15",
        "dense_question": "At what hour does the newly instituted transit route arrive at the institution's entrance?",
        "dense_answer": "7:15",
    },
]

for pair in VOCAB_PAIRS:
    compare_question_pair(
        pair["passage"],
        "PLAIN", pair["plain_question"], pair["plain_answer"],
        "DENSE", pair["dense_question"], pair["dense_answer"],
    )

PASSAGE: Lily always carries a spare phone charger in her bag. Her phone battery tends to die quickly during long days at work.

  PLAIN        Q: What does Lily always carry in her bag?
               gold='a spare phone charger'
               tok_entropy_norm=0.848  sent_entropy_norm=0.585  question_answer_overlap_coef=0.000
               predicted_answer='a spare phone charger'  confidence=0.714  f1=1.000  CORRECT

  DENSE        Q: What accessory does Lily routinely transport within her bag?
               gold='a spare phone charger'
               tok_entropy_norm=0.844  sent_entropy_norm=0.648  question_answer_overlap_coef=0.000
               predicted_answer='phone charger'  confidence=0.899  f1=0.667  CORRECT

  delta: tok_entropy_norm=-0.004  sent_entropy_norm=+0.062  overlap=+0.000  confidence=+0.185  f1=-0.333

PASSAGE: Every school day, Maria wakes up early. She needs to go to school at 9AM, so she prepares her books and uniform the night before. Her mother drops her of

<a id="manual-ratings"></a>
## 9. Rate difficulty manually

Workflow: set `IDX` above, look at the plot/text, then call `rate(IDX, score)`
below with a 1-5 difficulty score (1 = very easy, 5 = very hard). Each call
saves immediately to `question_difficulty/results/manual_difficulty_ratings.json`
(keyed by a stable hash of source+question, not by `IDX`, so ratings survive
across notebook restarts even if sampling changes).

In [ ]:
import hashlib
import json

RATINGS_PATH = Path("../results/manual_difficulty_ratings.json")
RATINGS_PATH.parent.mkdir(parents=True, exist_ok=True)

def _qid(rec):
    return hashlib.md5((rec["source"] + "::" + rec["question"]).encode()).hexdigest()[:12]

def _load_ratings():
    if RATINGS_PATH.exists():
        return json.loads(RATINGS_PATH.read_text())
    return {}

def rate(idx, score):
    """score: 1 (very easy) .. 5 (very hard)"""
    assert score in (1, 2, 3, 4, 5), "score must be 1-5"
    rec = results[idx]
    qid = _qid(rec)
    ratings = _load_ratings()
    ratings[qid] = {
        "source": rec["source"],
        "question": rec["question"],
        "entropy": rec["entropy"],
        "entropy_norm": rec["entropy_norm"],
        "num_sentences": rec["num_sentences"],
        "layer": rec["layer"],
        "qa_model": QA_MODEL,
        "manual_score": score,
    }
    RATINGS_PATH.write_text(json.dumps(ratings, indent=2, ensure_ascii=False))
    print(f"Saved: idx={idx} [{rec['source']}] score={score} entropy={rec['entropy']:.3f} entropy_norm={rec['entropy_norm']:.3f}")

def show_ratings():
    ratings = _load_ratings()
    print(f"{len(ratings)} ratings saved so far")
    for r in ratings.values():
        print(f"  score={r['manual_score']}  entropy={r['entropy']:.3f}  entropy_norm={r.get('entropy_norm', float('nan')):.3f}  [{r['source']}]  {r['question'][:60]}")

# Example usage after viewing IDX above:
# rate(IDX, 3)
show_ratings()